# Compilation & Audit Receipts

Every compilation run in qb-compiler produces a detailed audit trail: which passes ran, what changed, how long each step took, and what fidelity improvement was achieved. This notebook shows you how to use that information for reproducibility, debugging, and regulatory compliance.

This notebook covers:

1. Setting up `QBCompiler.from_backend()`
2. The `compiler.compile()` workflow
3. Inspecting `CompileResult` metrics
4. Analyzing the pass log (`PassResult` objects)
5. Generating compilation receipts as JSON audit trails
6. Comparing compilation across optimization strategies
7. Seed reproducibility
8. Budget guards (`BudgetExceededError`)
9. Saving and loading receipts for reproducibility

## 1. Compiler Setup

In [1]:
import json
import math
import time
from datetime import datetime, timezone

from qb_compiler import (
    BACKEND_CONFIGS,
    BudgetExceededError,
    CompileResult,
    PassResult,
    QBCircuit,
    QBCompiler,
)

In [2]:
# Create a compiler targeting IBM Fez with the default fidelity_optimal strategy
compiler = QBCompiler.from_backend("ibm_fez")

print(f"Strategy:           {compiler.strategy}")
print(f"Backend:            {compiler.config.backend}")
print(f"Optimization level: {compiler.config.optimization_level}")
print(f"Basis gates:        {compiler.config.effective_basis_gates}")

Strategy:           fidelity_optimal
Backend:            ibm_fez
Optimization level: 3
Basis gates:        ('id', 'rz', 'sx', 'x', 'cz', 'reset')


## 2. Basic Compilation Flow

Build a circuit, compile it, inspect the result.

In [3]:
# Build a 4-qubit circuit with some redundancy the optimizer can exploit
circ = QBCircuit(4)

# Layer 1: Hadamards
for q in range(4):
    circ.h(q)

# Layer 2: entangling
circ.cx(0, 1).cx(2, 3)

# Layer 3: rotations
circ.rz(0, math.pi / 4).rz(1, math.pi / 8)
circ.rz(2, math.pi / 4).rz(3, math.pi / 8)

# Layer 4: more entangling
circ.cx(1, 2)

# Layer 5: redundant H-H pair (should cancel)
circ.h(0)
circ.h(0)

# Layer 6: redundant CX-CX pair (should cancel)
circ.cx(0, 1)
circ.cx(0, 1)

circ.measure_all()

print(f"Input circuit: {circ}")
print(f"  Depth:      {circ.depth}")
print(f"  Gate count: {circ.gate_count}")
print(f"  2Q gates:   {circ.two_qubit_count}")

Input circuit: QBCircuit(n_qubits=4, ops=19, depth=8)
  Depth:      8
  Gate count: 19
  2Q gates:   5


In [4]:
result = compiler.compile(circ)

print(f"Compilation result:")
print(f"  Original depth:     {result.original_depth}")
print(f"  Compiled depth:     {result.compiled_depth}")
print(f"  Depth reduction:    {result.depth_reduction_pct:.1f}%")
print(f"  Estimated fidelity: {result.estimated_fidelity:.4f}")
print(f"  Compilation time:   {result.compilation_time_ms:.2f} ms")

Compilation result:
  Original depth:     8
  Compiled depth:     25
  Depth reduction:    -212.5%
  Estimated fidelity: 0.9524
  Compilation time:   4315.42 ms


## 3. Inspecting CompileResult

The `CompileResult` contains the compiled circuit, before/after metrics, and the full pass log.

In [5]:
# The compiled circuit is also a QBCircuit
compiled = result.compiled_circuit

print(f"Compiled circuit: {compiled}")
print(f"  Gate count:     {compiled.gate_count}")
print(f"  Gate names:     {compiled.gate_names}")
print()
print("Compiled operations:")
for op in compiled.ops:
    print(f"  {op.name:10s} qubits={op.qubits}  params={op.params}")

Compiled circuit: QBCircuit(n_qubits=24, ops=59, depth=25)
  Gate count:     59
  Gate names:     {'measure', 'sx', 'cz', 'rz'}

Compiled operations:
  rz         qubits=(21,)  params=(1.5707963267948966,)
  sx         qubits=(21,)  params=()
  rz         qubits=(21,)  params=(1.5707963267948966,)
  rz         qubits=(22,)  params=(1.5707963267948966,)
  sx         qubits=(22,)  params=()
  rz         qubits=(22,)  params=(1.5707963267948966,)
  rz         qubits=(23,)  params=(1.5707963267948966,)
  sx         qubits=(23,)  params=()
  rz         qubits=(23,)  params=(1.5707963267948966,)
  rz         qubits=(16,)  params=(1.5707963267948966,)
  sx         qubits=(16,)  params=()
  rz         qubits=(16,)  params=(1.5707963267948966,)
  rz         qubits=(22,)  params=(1.5707963267948966,)
  sx         qubits=(22,)  params=()
  rz         qubits=(22,)  params=(1.5707963267948966,)
  cz         qubits=(21, 22)  params=()
  rz         qubits=(22,)  params=(1.5707963267948966,)
  sx     

In [6]:
# Check if calibration-aware mapping was used
if result.initial_layout:
    print(f"Initial layout (logical -> physical): {result.initial_layout}")
    print(f"Layout list (for Qiskit transpile):   {result.initial_layout_list}")
else:
    print("No calibration-aware layout was applied (using identity mapping).")

Initial layout (logical -> physical): {1: 22, 2: 23, 0: 21, 3: 16}
Layout list (for Qiskit transpile):   [21, 22, 23, 16]


## 4. Pass Log Analysis

Each `PassResult` records what a single compiler pass did: gate count and depth before/after, and how long it took. This is essential for understanding where optimization time is spent.

In [7]:
print(f"{'Pass Name':25s}  {'Depth':>12s}  {'Gates':>12s}  {'Time (ms)':>10s}")
print("-" * 65)
for p in result.pass_log:
    print(
        f"{p.pass_name:25s}  "
        f"{p.depth_before:4d} -> {p.depth_after:<4d}  "
        f"{p.gate_count_before:4d} -> {p.gate_count_after:<4d}  "
        f"{p.elapsed_ms:8.3f}"
    )

Pass Name                         Depth         Gates   Time (ms)
-----------------------------------------------------------------
basis_translation             7 -> 26      19 -> 61       0.115
gate_cancellation            26 -> 26      61 -> 61       0.038
rotation_merge               26 -> 25      61 -> 59       0.090
gate_cancellation            25 -> 25      59 -> 59       0.019
rotation_merge               25 -> 25      59 -> 59       0.082


In [8]:
# Identify which passes actually changed the circuit
effective_passes = [
    p for p in result.pass_log
    if p.gate_count_before != p.gate_count_after or p.depth_before != p.depth_after
]
no_op_passes = [
    p for p in result.pass_log
    if p.gate_count_before == p.gate_count_after and p.depth_before == p.depth_after
]

print(f"Effective passes: {len(effective_passes)}")
for p in effective_passes:
    gates_removed = p.gate_count_before - p.gate_count_after
    print(f"  {p.pass_name}: removed {gates_removed} gates, depth {p.depth_before}->{p.depth_after}")

print(f"\nNo-op passes: {len(no_op_passes)}")
for p in no_op_passes:
    print(f"  {p.pass_name} ({p.elapsed_ms:.3f}ms)")

Effective passes: 2
  basis_translation: removed -42 gates, depth 7->26
  rotation_merge: removed 2 gates, depth 26->25

No-op passes: 3
  gate_cancellation (0.038ms)
  gate_cancellation (0.019ms)
  rotation_merge (0.082ms)


In [9]:
# Time distribution across passes
total_ms = sum(p.elapsed_ms for p in result.pass_log)

print(f"Total pass time: {total_ms:.2f} ms")
print(f"Overhead (compile - passes): {result.compilation_time_ms - total_ms:.2f} ms")
print()
print("Time distribution:")
for p in sorted(result.pass_log, key=lambda x: -x.elapsed_ms):
    pct = (p.elapsed_ms / total_ms * 100) if total_ms > 0 else 0
    bar = "#" * int(pct / 2)
    print(f"  {p.pass_name:25s} {p.elapsed_ms:8.3f}ms  {pct:5.1f}%  {bar}")

Total pass time: 0.34 ms
Overhead (compile - passes): 4315.08 ms

Time distribution:
  basis_translation            0.115ms   33.4%  ################
  rotation_merge               0.090ms   26.2%  #############
  rotation_merge               0.082ms   23.8%  ###########
  gate_cancellation            0.038ms   11.0%  #####
  gate_cancellation            0.019ms    5.5%  ##


## 5. Compilation Receipts

A compilation receipt is a JSON document that captures everything needed to reproduce or audit a compilation run. Let's build one.

In [10]:
def generate_receipt(result: CompileResult, compiler: QBCompiler, input_circuit: QBCircuit) -> dict:
    """Generate a JSON-serializable compilation receipt."""
    return {
        "receipt_version": "1.0",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "compiler": {
            "backend": compiler.config.backend,
            "strategy": compiler.strategy,
            "optimization_level": compiler.config.optimization_level,
            "basis_gates": list(compiler.config.effective_basis_gates),
        },
        "input_circuit": {
            "n_qubits": input_circuit.n_qubits,
            "gate_count": input_circuit.gate_count,
            "depth": input_circuit.depth,
            "two_qubit_gates": input_circuit.two_qubit_count,
        },
        "output": {
            "compiled_depth": result.compiled_depth,
            "original_depth": result.original_depth,
            "depth_reduction_pct": round(result.depth_reduction_pct, 2),
            "estimated_fidelity": result.estimated_fidelity,
            "compilation_time_ms": result.compilation_time_ms,
            "initial_layout": result.initial_layout,
        },
        "pass_log": [
            {
                "pass_name": p.pass_name,
                "depth_before": p.depth_before,
                "depth_after": p.depth_after,
                "gate_count_before": p.gate_count_before,
                "gate_count_after": p.gate_count_after,
                "elapsed_ms": p.elapsed_ms,
                "detail": p.detail,
            }
            for p in result.pass_log
        ],
    }


receipt = generate_receipt(result, compiler, circ)
print(json.dumps(receipt, indent=2))

{
  "receipt_version": "1.0",
  "timestamp": "2026-06-12T11:31:36.023186+00:00",
  "compiler": {
    "backend": "ibm_fez",
    "strategy": "fidelity_optimal",
    "optimization_level": 3,
    "basis_gates": [
      "id",
      "rz",
      "sx",
      "x",
      "cz",
      "reset"
    ]
  },
  "input_circuit": {
    "n_qubits": 4,
    "gate_count": 19,
    "depth": 8,
    "two_qubit_gates": 5
  },
  "output": {
    "compiled_depth": 25,
    "original_depth": 8,
    "depth_reduction_pct": -212.5,
    "estimated_fidelity": 0.9524292791933862,
    "compilation_time_ms": 4315.424,
    "initial_layout": {
      "1": 22,
      "2": 23,
      "0": 21,
      "3": 16
    }
  },
  "pass_log": [
    {
      "pass_name": "basis_translation",
      "depth_before": 7,
      "depth_after": 26,
      "gate_count_before": 19,
      "gate_count_after": 61,
      "elapsed_ms": 0.115,
      "detail": ""
    },
    {
      "pass_name": "gate_cancellation",
      "depth_before": 26,
      "depth_after": 26,

## 6. Comparing Optimization Strategies

qb-compiler supports three strategies with different optimization/cost trade-offs:

| Strategy | Level | Optimization | Use case |
|----------|-------|-------------|----------|
| `fidelity_optimal` | 3 | Maximum fidelity | Production, critical experiments |
| `depth_optimal` | 2 | Minimum depth | Latency-sensitive circuits |
| `budget_optimal` | 1 | Fast compilation | Low-cost backends, rapid iteration |

In [11]:
# Build a circuit with enough structure to show strategy differences
circ = QBCircuit(5)
for q in range(5):
    circ.h(q)
circ.cx(0, 1).cx(1, 2).cx(2, 3).cx(3, 4)
for q in range(5):
    circ.rz(q, math.pi / 3)
circ.cx(0, 1).cx(1, 2).cx(2, 3).cx(3, 4)
# Add redundant gates
circ.h(0).h(0)  # cancels
circ.cx(2, 3).cx(2, 3)  # cancels
circ.x(4).x(4)  # cancels
circ.measure_all()

print(f"Input: depth={circ.depth}, gates={circ.gate_count}, 2Q={circ.two_qubit_count}")
print()

Input: depth=11, gates=29, 2Q=10



In [12]:
print(f"{'Strategy':20s}  {'Depth':>6s}  {'Gates':>6s}  {'Fidelity':>10s}  {'Time (ms)':>10s}  {'Reduction':>10s}")
print("-" * 70)

receipts = {}
for strategy in ["fidelity_optimal", "depth_optimal", "budget_optimal"]:
    r = compiler.compile(circ, strategy=strategy)
    receipts[strategy] = generate_receipt(r, compiler, circ)
    print(
        f"{strategy:20s}  "
        f"{r.compiled_depth:6d}  "
        f"{r.compiled_circuit.gate_count:6d}  "
        f"{r.estimated_fidelity:10.4f}  "
        f"{r.compilation_time_ms:10.2f}  "
        f"{r.depth_reduction_pct:9.1f}%"
    )

Strategy               Depth   Gates    Fidelity   Time (ms)   Reduction
----------------------------------------------------------------------


fidelity_optimal          41      99      0.9241     1351.69     -272.7%


depth_optimal             41      99      0.9241     1394.61     -272.7%
budget_optimal            43     101      0.8664        0.62     -290.9%


In [13]:
# Compare pass logs between strategies
for strategy in ["fidelity_optimal", "budget_optimal"]:
    print(f"\n=== {strategy} pass log ===")
    for p in receipts[strategy]["pass_log"]:
        print(
            f"  {p['pass_name']:25s}  "
            f"gates {p['gate_count_before']:3d}->{p['gate_count_after']:<3d}  "
            f"{p['elapsed_ms']:.3f}ms"
        )


=== fidelity_optimal pass log ===
  basis_translation          gates  29->103  0.079ms
  gate_cancellation          gates 103->101  0.074ms
  rotation_merge             gates 101->99   0.153ms
  gate_cancellation          gates  99->99   0.031ms
  rotation_merge             gates  99->99   0.161ms

=== budget_optimal pass log ===
  basis_translation          gates  29->103  0.072ms
  gate_cancellation          gates 103->101  0.101ms


## 7. Comparing Compilation Across Backends

Different backends have different basis gates and coupling maps, so the same circuit compiles differently.

In [14]:
circ = QBCircuit(4)
for q in range(4):
    circ.h(q)
circ.cx(0, 1).cx(1, 2).cx(2, 3)
for q in range(4):
    circ.rz(q, math.pi / 4)
circ.measure_all()

print(f"Input: depth={circ.depth}, gates={circ.gate_count}")
print()

print(f"{'Backend':18s}  {'Depth':>6s}  {'Reduction':>10s}  {'Fidelity':>10s}  {'Time (ms)':>10s}")
print("-" * 60)

for backend in ["ibm_fez", "ibm_torino", "rigetti_ankaa", "ionq_aria", "iqm_garnet"]:
    comp = QBCompiler.from_backend(backend)
    r = comp.compile(circ)
    print(
        f"{backend:18s}  "
        f"{r.compiled_depth:6d}  "
        f"{r.depth_reduction_pct:9.1f}%  "
        f"{r.estimated_fidelity:10.4f}  "
        f"{r.compilation_time_ms:10.2f}"
    )

Input: depth=6, gates=15

Backend              Depth   Reduction    Fidelity   Time (ms)
------------------------------------------------------------
ibm_fez                 19     -216.7%      0.9586       57.24
ibm_torino              20     -233.3%      0.9207       39.65
rigetti_ankaa           14     -133.3%      0.8271       48.84


ionq_aria                6        0.0%      0.9786     7813.29
iqm_garnet               9      -50.0%      0.7307       60.28


## 8. Seed Reproducibility

For reproducible results, the pass manager uses deterministic algorithms. Compiling the same circuit with the same configuration should produce the same output.

In [15]:
# Compile the same circuit three times
circ = QBCircuit(3).h(0).cx(0, 1).cx(1, 2).measure_all()

results_repeated = []
for i in range(3):
    r = compiler.compile(circ)
    results_repeated.append(r)
    print(
        f"Run {i+1}: depth={r.compiled_depth}, "
        f"fidelity={r.estimated_fidelity:.4f}, "
        f"gates={r.compiled_circuit.gate_count}"
    )

# Verify determinism
depths = [r.compiled_depth for r in results_repeated]
fidelities = [r.estimated_fidelity for r in results_repeated]

print(f"\nDeterministic: {len(set(depths)) == 1 and len(set(fidelities)) == 1}")

Run 1: depth=11, fidelity=0.9742, gates=20
Run 2: depth=11, fidelity=0.9742, gates=20
Run 3: depth=11, fidelity=0.9742, gates=20

Deterministic: True


## 9. Budget Guard

The `budget_usd` parameter on `compile()` protects you from accidentally submitting expensive jobs. If the estimated cost at 1024 shots exceeds the budget, a `BudgetExceededError` is raised before the result is returned.

In [16]:
circ = QBCircuit(3).h(0).cx(0, 1).cx(1, 2).measure_all()

# This should succeed -- Bell state is cheap on IBM
result = compiler.compile(circ, budget_usd=1.00)
print(f"Compiled within budget: fidelity={result.estimated_fidelity:.4f}")

# This will raise -- budget is impossibly small
try:
    compiler.compile(circ, budget_usd=0.000001)
except BudgetExceededError as e:
    print(f"\nBudget exceeded: {e}")

Compiled within budget: fidelity=0.9742

Budget exceeded: Estimated cost $0.1638 exceeds budget $0.0000 (1024 shots)


In [17]:
# Budget guard is especially useful with expensive backends
compiler_ionq = QBCompiler.from_backend("ionq_aria")

circ_small = QBCircuit(2).h(0).cx(0, 1).measure_all()

try:
    compiler_ionq.compile(circ_small, budget_usd=0.50)
except BudgetExceededError as e:
    print(f"IonQ budget guard: {e}")

# With a larger budget, it succeeds
r = compiler_ionq.compile(circ_small, budget_usd=500.00)
print(f"\nCompiled on IonQ: fidelity={r.estimated_fidelity:.4f}")

IonQ budget guard: Estimated cost $307.2000 exceeds budget $0.5000 (1024 shots)



Compiled on IonQ: fidelity=0.9876


## 10. Cost Estimation

Use `estimate_cost()` to project costs at different shot counts.

In [18]:
circ = QBCircuit(4)
for q in range(4):
    circ.h(q)
circ.cx(0, 1).cx(1, 2).cx(2, 3)
circ.measure_all()

result = compiler.compile(circ)
compiled = result.compiled_circuit

print(f"{'Shots':>8s}  {'Cost':>10s}  {'Per-shot':>12s}  {'Within $1':>10s}  {'Within $10':>10s}")
print("-" * 55)
for shots in [100, 1_024, 4_096, 10_000, 100_000]:
    cost = compiler.estimate_cost(compiled, shots=shots)
    print(
        f"{shots:>8,}  "
        f"${cost.total_usd:>8.4f}  "
        f"${cost.cost_per_shot_usd:>10.6f}  "
        f"{'yes' if cost.within_budget(1.0) else 'NO':>10s}  "
        f"{'yes' if cost.within_budget(10.0) else 'NO':>10s}"
    )

   Shots        Cost      Per-shot   Within $1  Within $10
-------------------------------------------------------
     100  $  0.0160  $  0.000160         yes         yes
   1,024  $  0.1638  $  0.000160         yes         yes
   4,096  $  0.6554  $  0.000160         yes         yes
  10,000  $  1.6000  $  0.000160          NO         yes
 100,000  $ 16.0000  $  0.000160          NO          NO


## 11. Saving and Loading Receipts

Save receipts to disk for auditability. Load them later to reproduce or compare results.

In [19]:
import tempfile
import os

# Generate a receipt
circ = QBCircuit(4).h(0).cx(0, 1).cx(1, 2).cx(2, 3).measure_all()
result = compiler.compile(circ)
receipt = generate_receipt(result, compiler, circ)

# Save to a temp file
receipt_path = os.path.join(tempfile.gettempdir(), "compilation_receipt.json")
with open(receipt_path, "w") as f:
    json.dump(receipt, f, indent=2)
print(f"Receipt saved to: {receipt_path}")

# Load it back
with open(receipt_path) as f:
    loaded = json.load(f)

print(f"\nLoaded receipt:")
print(f"  Backend:          {loaded['compiler']['backend']}")
print(f"  Strategy:         {loaded['compiler']['strategy']}")
print(f"  Compiled depth:   {loaded['output']['compiled_depth']}")
print(f"  Fidelity:         {loaded['output']['estimated_fidelity']:.4f}")
print(f"  Passes run:       {len(loaded['pass_log'])}")
print(f"  Timestamp:        {loaded['timestamp']}")

Receipt saved to: /tmp/compilation_receipt.json

Loaded receipt:
  Backend:          ibm_fez
  Strategy:         fidelity_optimal
  Compiled depth:   15
  Fidelity:         0.9620
  Passes run:       5
  Timestamp:        2026-06-12T11:31:47.980021+00:00


In [20]:
# Compare two receipts side by side
# Compile the same circuit on two backends and compare receipts
circ = QBCircuit(4).h(0).cx(0, 1).cx(1, 2).cx(2, 3).measure_all()

r_fez = QBCompiler.from_backend("ibm_fez").compile(circ)
r_torino = QBCompiler.from_backend("ibm_torino").compile(circ)

rec_fez = generate_receipt(r_fez, QBCompiler.from_backend("ibm_fez"), circ)
rec_torino = generate_receipt(r_torino, QBCompiler.from_backend("ibm_torino"), circ)

print(f"{'Metric':25s}  {'IBM Fez':>12s}  {'IBM Torino':>12s}")
print("-" * 55)
for key in ["compiled_depth", "depth_reduction_pct", "estimated_fidelity", "compilation_time_ms"]:
    v1 = rec_fez["output"][key]
    v2 = rec_torino["output"][key]
    fmt = ".4f" if isinstance(v1, float) else "d"
    print(f"{key:25s}  {v1:>12{fmt}}  {v2:>12{fmt}}")

Metric                          IBM Fez    IBM Torino
-------------------------------------------------------
compiled_depth                       15            16
depth_reduction_pct           -200.0000     -220.0000
estimated_fidelity               0.9620        0.9246
compilation_time_ms            570.1830       50.9920


## 12. Compiling Larger Circuits

Let's compile a more realistic circuit and see how the pass log looks for a deeper circuit with more optimization opportunities.

In [21]:
# Build a 6-qubit circuit with multiple optimization opportunities
circ = QBCircuit(6)

# Layer 1: initial superposition
for q in range(6):
    circ.h(q)

# Layers 2-4: entangling + rotations (realistic VQE-like structure)
for layer in range(3):
    for q in range(5):
        circ.cx(q, q + 1)
    for q in range(6):
        circ.rz(q, math.pi / (2 + layer))
        circ.rx(q, math.pi / (3 + layer))

# Redundant gates that optimization should catch
circ.cx(0, 1).cx(0, 1)  # cancels
circ.h(3).h(3)  # cancels
circ.x(5).x(5)  # cancels

circ.measure_all()

print(f"Input: depth={circ.depth}, gates={circ.gate_count}, 2Q={circ.two_qubit_count}")

result = compiler.compile(circ)
print(f"Output: depth={result.compiled_depth}, fidelity={result.estimated_fidelity:.4f}")
print(f"Depth reduction: {result.depth_reduction_pct:.1f}%")
print()
print("Pass log:")
for p in result.pass_log:
    delta_gates = p.gate_count_after - p.gate_count_before
    sign = "+" if delta_gates >= 0 else ""
    print(f"  {p.pass_name:25s}  gates: {sign}{delta_gates}  depth: {p.depth_before}->{p.depth_after}")

Input: depth=19, gates=69, 2Q=17


Output: depth=48, fidelity=0.8878
Depth reduction: -152.6%

Pass log:
  basis_translation          gates: +118  depth: 18->50
  gate_cancellation          gates: -2  depth: 50->48
  rotation_merge             gates: -2  depth: 48->48
  gate_cancellation          gates: +0  depth: 48->48
  rotation_merge             gates: +0  depth: 48->48


## Summary

In this notebook you learned:

- **`QBCompiler.from_backend()`**: create a compiler configured for any supported backend
- **`compiler.compile()`**: compile a circuit and get a `CompileResult` with full metrics
- **`CompileResult`**: `compiled_depth`, `depth_reduction_pct`, `estimated_fidelity`, `compilation_time_ms`, `initial_layout`
- **Pass log**: iterate `PassResult` objects to see exactly what each optimization pass did
- **Receipts**: generate JSON audit trails for reproducibility and compliance
- **Strategies**: `fidelity_optimal`, `depth_optimal`, `budget_optimal` trade off quality vs speed
- **Seed reproducibility**: same input produces same output
- **Budget guard**: `BudgetExceededError` prevents expensive accidents
- **Cost estimation**: `estimate_cost()` projects costs at any shot count

Next: [03: Multi-Vendor Backend Ranking](03_multi_vendor_ranking.ipynb)